# AC_TC_2B Processing — v2.0

Grids **all tropospheric cloud occurrence** from the AC__TC__2B product onto a
global 1° × 100 m grid.

## What is new in v2 compared to v1.1

| | v1.1 | v2 |
|---|---|---|
| Target classes | Ice only (3, 13, 14, 15, 19, 21) | All tropospheric clouds (codes 7–21) |
| MAX_QC_FLAG | 4 (effectively no filter) | 1 (Good + Likely Good only) |
| Output variable naming | `ice_count` / `occurrence` | `cloud_count` / `occurrence` |

Codes 7–21 cover all hydrometeor targets identified by the synergetic algorithm
excluding heavy rain (5) and heavy mixed-phase precipitation (6), which are not
represented in ATL_TC and would make the comparison unfair.

## Workflow
1. Run **Cells 0–4** (imports, ZIP helpers, config, discovery, grid).
2. Run **Cell 5** to define the accumulator function.
3. Run **Cell 6** (5-ZIP test) and sanity-check the printed pixel counts.
4. Run **Cells 7–9** for the full batch and save.

**Runs on:** Work PC (needs access to remote AC_TC data).

## 0. Imports

In [ ]:
import earthcarekit as eck
from pathlib import Path

_cfg = Path("/usr/people/raucher/Documents/Config_ECK/default_config.toml")
if _cfg.exists():
    eck.set_config(str(_cfg))

import xarray as xr
import numpy as np
import re
import pandas as pd
import sys, shlex, shutil
from datetime import datetime, timedelta, date
from IPython.display import display
import matplotlib.pyplot as plt
import os
from scipy.interpolate import interp1d

MY_SUBPROCESS_FILE = Path("/usr/people/raucher/Documents/Coding1/Gerd-Jan/OneDrive_1_24-2-2026")
if str(MY_SUBPROCESS_FILE) not in sys.path:
    sys.path.insert(0, str(MY_SUBPROCESS_FILE))
from my_subprocess import run_shell_cmd_and_communicate, print_shell_output

print("Imports OK")

## 0b. ZIP helpers

In [ ]:
def _to_date(v):
    if isinstance(v, datetime): return v.date()
    if isinstance(v, date):     return v
    if isinstance(v, str):      return datetime.strptime(v, "%Y-%m-%d").date()
    raise TypeError(f"Unsupported date type: {type(v)}")


def run_cmd_checked(cmd: str, verbose: bool = False):
    lines_out, lines_err, rc = run_shell_cmd_and_communicate(cmd, verbose=verbose)
    if rc != 0:
        print_shell_output(lines_out, lines_err, prefix="[shell] ")
        raise RuntimeError(f"Command failed (exit {rc}): {cmd}")
    return lines_out, lines_err


def discover_remote_zip_files(remote_product_root, start_date, end_date):
    root  = Path(remote_product_root)
    start = _to_date(start_date)
    end   = _to_date(end_date)
    if end < start:
        raise ValueError("end_date must be >= start_date")
    zips, day = [], start
    while day <= end:
        day_dir = root / day.strftime("%Y") / day.strftime("%m") / day.strftime("%d")
        if day_dir.exists():
            zips.extend(sorted(day_dir.glob("*.ZIP")))
            zips.extend(sorted(day_dir.glob("*.zip")))
        day += timedelta(days=1)
    return sorted(dict.fromkeys(zips))


def stage_zip_and_extract(src_zip: Path, local_stage_root: Path):
    local_stage_root.mkdir(parents=True, exist_ok=True)
    local_zip   = local_stage_root / src_zip.name
    extract_dir = local_stage_root / src_zip.stem
    if local_zip.exists():   local_zip.unlink()
    if extract_dir.exists(): shutil.rmtree(extract_dir)
    run_cmd_checked(f"cp {shlex.quote(str(src_zip))} {shlex.quote(str(local_zip))}")
    run_cmd_checked(f"unzip -oq {shlex.quote(str(local_zip))} -d {shlex.quote(str(extract_dir))}")
    h5_files = sorted(extract_dir.rglob("*.h5"))
    if not h5_files:
        raise FileNotFoundError(f"No .h5 after extracting: {src_zip}")
    return local_zip, extract_dir, h5_files


def cleanup_staged_data(local_zip: Path | None, extract_dir: Path | None):
    if extract_dir is not None and extract_dir.exists(): shutil.rmtree(extract_dir)
    if local_zip   is not None and local_zip.exists():   local_zip.unlink()


print("ZIP helpers defined")

## 1. Config

All tunable parameters live here.  Grid settings must match ATL_TC v2 so
the two products can be compared directly.

In [ ]:
# ── Remote data paths ─────────────────────────────────────────────────────────
REMOTE_PRODUCT_ROOT = Path("/net/pc230016/nobackup_1/users/zadelhof/EarthCARE_DATA/L2/AC__TC__2B")

# ── Local staging folder (safe to clean up) ───────────────────────────────────
LOCAL_STAGE_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/temp_ZIPs_2B")
LOCAL_STAGE_ROOT.mkdir(parents=True, exist_ok=True)

# ── Date range ────────────────────────────────────────────────────────────────
START_DATE = "2025-01-01"   # YYYY-MM-DD
END_DATE   = "2026-02-28"   # YYYY-MM-DD

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT = Path("/usr/people/raucher/Documents/Coding1/KNMI/KNMI/ACTC_output")

# ── Classification settings ───────────────────────────────────────────────────
CLASS_VAR = "synergetic_target_classification"

# All tropospheric cloud codes (7–21), excluding heavy precipitation (5–6):
#    7 = no rain or ice (possible liquid)
#    8 = liquid cloud
#    9 = drizzling liquid cloud
#   10 = warm rain
#   11 = cold rain
#   12 = melting snow
#   13 = snow (possible liquid)
#   14 = snow (no liquid)
#   15 = rimed snow (possible liquid)
#   16 = rimed snow and supercooled liquid
#   17 = snow and supercooled liquid
#   18 = supercooled liquid
#   19 = ice cloud (possible liquid)
#   20 = ice and supercooled liquid
#   21 = ice cloud (no liquid)
TARGET_CLASSES = list(range(7, 22))

# Codes always excluded from denominator:
#   -1 = unknown    0 = ground
EXCLUDE_CODES = [-1, 0]

# Quality filter: pixels with quality_status > MAX_QC_FLAG are excluded.
# 0=Good  1=Likely Good  2=Likely Bad  3=Bad/attenuated  4=Missing L1
QC_VAR      = "quality_status"
MAX_QC_FLAG = 1

# ── Coordinate variables ──────────────────────────────────────────────────────
LAT        = "latitude"
LON        = "longitude"
HEIGHT_VAR = "height"

# ── Grid settings ─────────────────────────────────────────────────────────────
GRID_RES_DEG         = 1.0
MAX_HEIGHT_M         = 20_000.0
MIN_SAMPLES_PER_CELL = 10

start_dt = datetime.strptime(START_DATE, "%Y-%m-%d").date()
end_dt   = datetime.strptime(END_DATE,   "%Y-%m-%d").date()
if end_dt < start_dt:
    raise ValueError("END_DATE must be >= START_DATE")

if not REMOTE_PRODUCT_ROOT.exists():
    raise FileNotFoundError(f"Remote path not mounted/reachable: {REMOTE_PRODUCT_ROOT}")

print("Config loaded")
print(f"Product:        AC__TC__2B")
print(f"Remote root:    {REMOTE_PRODUCT_ROOT}")
print(f"Stage root:     {LOCAL_STAGE_ROOT}")
print(f"Date range:     {start_dt} to {end_dt}")
print(f"CLASS_VAR:      {CLASS_VAR!r}")
print(f"TARGET_CLASSES: {TARGET_CLASSES}  (all tropospheric clouds)")
print(f"EXCLUDE_CODES:  {EXCLUDE_CODES}   (unknown + ground, always excluded from denominator)")
print(f"MAX_QC_FLAG:    {MAX_QC_FLAG}")
print(f"MAX_HEIGHT_M:   {MAX_HEIGHT_M:.0f} m  |  GRID_RES_DEG: {GRID_RES_DEG}")

## 2. File Discovery

In [ ]:
zip_paths = discover_remote_zip_files(REMOTE_PRODUCT_ROOT, start_dt, end_dt)

print(f"Discovered {len(zip_paths)} ZIP files  |  Range: {start_dt} -> {end_dt}")
for p in zip_paths[:3]:
    print("  -", p)
if len(zip_paths) > 3:
    print(f"  ... and {len(zip_paths)-3} more")

if len(zip_paths) == 0:
    raise FileNotFoundError("No ZIP files found for this date range/product.")

## 3. Single-File Inspection

Open the first file to verify the product structure and check that the target
cloud codes (7–21) are present.

In [ ]:
local_zip = None
extract_dir = None

try:
    local_zip, extract_dir, staged_h5 = stage_zip_and_extract(zip_paths[0], LOCAL_STAGE_ROOT)
    fp0 = str(staged_h5[0])
    print("Inspecting staged file:", fp0)

    with eck.read_product(fp0) as ds0:
        print("\nDataset summary:")
        print("height shape:", ds0[HEIGHT_VAR].shape)
        display(ds0)

        required_vars = [CLASS_VAR, LAT, HEIGHT_VAR, LON]
        print("\nRequired variable check:")
        for v in required_vars:
            print(f" - {v}: {'OK' if v in ds0.data_vars else 'MISSING'}")

        print("\nClassification metadata:")
        print("long_name:", ds0[CLASS_VAR].attrs.get("long_name", "n/a"))
        definition = ds0[CLASS_VAR].attrs.get("definition", "n/a")
        print("definition:\n", definition)

        # Parse code -> meaning from the definition text
        code_to_meaning = {}
        for line in definition.splitlines():
            line = line.strip()
            m = re.match(r"^(-?\d+)\s*:\s*(.+)$", line)
            if m:
                code_to_meaning[int(m.group(1))] = m.group(2)

        cls_vals = ds0[CLASS_VAR].values
        unique_codes = (
            pd.Series(cls_vals.ravel())
            .dropna()
            .astype(int)
            .value_counts()
            .sort_index()
            .rename("count")
            .reset_index()
            .rename(columns={"index": "target_class_code"})
        )
        unique_codes.insert(0, "target_class_name",
            unique_codes["target_class_code"].map(code_to_meaning).fillna("Unknown"))
        unique_codes["is_cloud_target"] = unique_codes["target_class_code"].isin(TARGET_CLASSES)
        print("\nClass codes (name | code | count | is_cloud_target):")
        display(unique_codes)
finally:
    cleanup_staged_data(local_zip, extract_dir)

## 4. Grid Setup

In [ ]:
# Global lat/lon bins
lat_bins = np.arange(-90.0,  90.0  + GRID_RES_DEG, GRID_RES_DEG)
lon_bins = np.arange(-180.0, 180.0 + GRID_RES_DEG, GRID_RES_DEG)

lat_centers = 0.5 * (lat_bins[:-1] + lat_bins[1:])
lon_centers = 0.5 * (lon_bins[:-1] + lon_bins[1:])

n_lat = len(lat_centers)
n_lon = len(lon_centers)

# Fixed height axis: 100 m spacing from 0 to MAX_HEIGHT_M
# JSG_height=242 confirmed in AC__TC__2B; matches the 100 m ATLID JSG grid.
target_h = np.arange(0.0, MAX_HEIGHT_M + 1.0, 100.0)   # 201 levels: 0..20000 m
n_height  = target_h.size

# 3D accumulators: tropospheric cloud occurrence
total_count_3d = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
cloud_count_3d = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)

print("Grid ready")
print(f"Shape (lat, lon, height): {total_count_3d.shape}")
print(f"Height range: {float(np.nanmin(target_h)):.0f} .. {float(np.nanmax(target_h)):.0f} m")
print(f"Latitude centers:  {lat_centers[0]} .. {lat_centers[-1]}")
print(f"Longitude centers: {lon_centers[0]} .. {lon_centers[-1]}")

## 5. One-File Accumulator

Valid pixels (denominator): finite height & classification, code not in `EXCLUDE_CODES`
(-1=unknown, 0=ground), quality_status ≤ MAX_QC_FLAG.

Cloud pixels (numerator): code in `TARGET_CLASSES` = 7–21.

In [ ]:
def accumulate_one_file(fp, lat_bins, lon_bins, target_h):
    with eck.read_product(str(fp)) as ds:
        cls = ds[CLASS_VAR].values.astype(float)
        h   = ds[HEIGHT_VAR].values
        lat = ds[LAT].values
        lon = ds[LON].values
        has_qc = QC_VAR in ds.data_vars
        qc     = ds[QC_VAR].values.astype(int) if has_qc else None

        n_obs    = cls.shape[0]
        n_height = target_h.size
        cloud_interp = np.full((n_obs, n_height), np.nan, dtype=float)

        for i in range(n_obs):
            h_i   = h[i, :]
            cls_i = cls[i, :]

            qc_ok = (qc[i, :] <= MAX_QC_FLAG) if has_qc else True
            valid = (
                np.isfinite(h_i) & np.isfinite(cls_i)
                & ~np.isin(cls_i, EXCLUDE_CODES)
                & qc_ok
            )
            if np.sum(valid) < 2:
                continue

            h_v     = h_i[valid]
            cloud_v = np.isin(cls_i[valid], TARGET_CLASSES).astype(float)

            order = np.argsort(h_v)
            h_v, cloud_v = h_v[order], cloud_v[order]
            h_v, idx     = np.unique(h_v, return_index=True)
            cloud_v      = cloud_v[idx]
            if h_v.size < 2:
                continue

            cloud_interp[i, :] = interp1d(
                h_v, cloud_v, kind="nearest",
                bounds_error=False, fill_value=np.nan
            )(target_h)

    total_hist_3d = np.zeros((len(lat_bins)-1, len(lon_bins)-1, n_height), dtype=np.float64)
    cloud_hist_3d = np.zeros_like(total_hist_3d)

    lat2d = np.broadcast_to(lat[:, None], cloud_interp.shape)
    lon2d = np.broadcast_to(lon[:, None], cloud_interp.shape)

    for k in range(n_height):
        v = cloud_interp[:, k]
        m = np.isfinite(v) & np.isfinite(lat2d[:, k]) & np.isfinite(lon2d[:, k])
        if not np.any(m):
            continue
        total_k, _, _ = np.histogram2d(
            lat2d[:, k][m], lon2d[:, k][m], bins=[lat_bins, lon_bins]
        )
        cloud_k, _, _ = np.histogram2d(
            lat2d[:, k][m], lon2d[:, k][m], bins=[lat_bins, lon_bins],
            weights=(v[m] == 1).astype(float)
        )
        total_hist_3d[:, :, k] = total_k
        cloud_hist_3d[:, :, k] = cloud_k

    return total_hist_3d, cloud_hist_3d


### TEST ###
local_zip = None
extract_dir = None
try:
    local_zip, extract_dir, staged_h5 = stage_zip_and_extract(zip_paths[0], LOCAL_STAGE_ROOT)
    t1, c1 = accumulate_one_file(staged_h5[0], lat_bins, lon_bins, target_h)
    print("one-file total pixels: ", int(np.nansum(t1)))
    print("one-file cloud pixels: ", int(np.nansum(c1)))
    print("shape:", t1.shape)
finally:
    cleanup_staged_data(local_zip, extract_dir)

## 6. Few-File Test Run

In [ ]:
N_QUICK_TEST = min(5, len(zip_paths))

total_test = np.zeros_like(total_count_3d)
cloud_test = np.zeros_like(cloud_count_3d)
failed_test       = []
processed_h5_test = 0

for src_zip in zip_paths[:N_QUICK_TEST]:
    local_zip = None
    extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ROOT)
        for fp in h5_files:
            t3d, c3d = accumulate_one_file(fp, lat_bins, lon_bins, target_h)
            total_test += t3d
            cloud_test += c3d
            processed_h5_test += 1
    except Exception as e:
        failed_test.append((str(src_zip), str(e)))
    finally:
        cleanup_staged_data(local_zip, extract_dir)

print(f"Tested ZIPs: {N_QUICK_TEST}  |  H5 processed: {processed_h5_test}  |  Failed: {len(failed_test)}")
print("Total valid pixels: ", int(np.nansum(total_test)))
print("Total cloud pixels: ", int(np.nansum(cloud_test)))
if np.nansum(total_test) > 0:
    print("Overall cloud occurrence (test):", float(np.nansum(cloud_test) / np.nansum(total_test)))

if failed_test:
    print("\nFirst failure:", failed_test[0][1])

# Quick look
num2d = np.nansum(cloud_test, axis=2)
den2d = np.nansum(total_test, axis=2)
occ2d = np.divide(num2d, den2d, out=np.full_like(num2d, np.nan), where=den2d > 0)

plt.figure(figsize=(8, 4))
plt.imshow(occ2d, origin="lower", vmin=0, vmax=1)
plt.colorbar(label="Cloud occurrence")
plt.title("Quick test — AC__TC__2B tropospheric cloud occurrence (height-collapsed)")
plt.xlabel("lon bin")
plt.ylabel("lat bin")
plt.show()

## 7. Full Batch Processing

In [ ]:
# Re-initialise accumulators (safe to re-run)
total_count_3d = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)
cloud_count_3d = np.zeros((n_lat, n_lon, n_height), dtype=np.float64)

failed_zips  = []
processed_h5 = 0
n_zips       = len(zip_paths)

for idx, src_zip in enumerate(zip_paths, start=1):
    local_zip = None
    extract_dir = None
    try:
        local_zip, extract_dir, h5_files = stage_zip_and_extract(src_zip, LOCAL_STAGE_ROOT)
        for fp in h5_files:
            t3d, c3d = accumulate_one_file(fp, lat_bins, lon_bins, target_h)
            total_count_3d += t3d
            cloud_count_3d += c3d
            processed_h5   += 1
    except Exception as e:
        failed_zips.append((str(src_zip), str(e)))
    finally:
        cleanup_staged_data(local_zip, extract_dir)

    if idx % 10 == 0 or idx == n_zips:
        print(f"{idx}/{n_zips} ZIPs | h5={processed_h5} | failed={len(failed_zips)}")

print("\nBatch done")
print(f"H5 files processed: {processed_h5}  |  ZIP failures: {len(failed_zips)}")
tot_sum = np.nansum(total_count_3d)
if tot_sum > 0:
    print(f"Total valid pixels:        {int(tot_sum)}")
    print(f"Overall cloud occurrence:  {float(np.nansum(cloud_count_3d)/tot_sum):.4f}")

if failed_zips:
    print("\nFirst 3 failures:")
    for z, err in failed_zips[:3]:
        print(" -", z)
        print("   ", err)

## 8. Occurrence Map

In [ ]:
# Height-collapsed 2D map (lat, lon)
cloud_2d = np.nansum(cloud_count_3d, axis=2)
tot_2d   = np.nansum(total_count_3d, axis=2)

occurrence_2d = np.divide(
    cloud_2d, tot_2d,
    out=np.full_like(cloud_2d, np.nan, dtype=np.float64),
    where=tot_2d > 0,
)
if MIN_SAMPLES_PER_CELL and MIN_SAMPLES_PER_CELL > 0:
    occurrence_2d[tot_2d < MIN_SAMPLES_PER_CELL] = np.nan

# Lat/height map (longitude collapsed)
cloud_lh = np.nansum(cloud_count_3d, axis=1)
tot_lh   = np.nansum(total_count_3d, axis=1)

occurrence_lh = np.divide(
    cloud_lh, tot_lh,
    out=np.full_like(cloud_lh, np.nan, dtype=np.float64),
    where=tot_lh > 0,
)
if MIN_SAMPLES_PER_CELL and MIN_SAMPLES_PER_CELL > 0:
    occurrence_lh[tot_lh < MIN_SAMPLES_PER_CELL] = np.nan

print("Occurrence computed")
print(f"2D (lat/lon):    {occurrence_2d.shape}")
print(f"2D (lat/height): {occurrence_lh.shape}")
tot_sum = np.nansum(total_count_3d)
if tot_sum > 0:
    print(f"Global cloud occurrence: {float(np.nansum(cloud_count_3d)/tot_sum):.6f}")

## 9. Save to NetCDF

In [ ]:
outdir = f"{OUTPUT_ROOT}/{start_dt:%Y%m%d}_{end_dt:%Y%m%d}"
os.makedirs(outdir, exist_ok=True)

_base = f"AC_TC_2B_v2_{GRID_RES_DEG}deg_{start_dt:%Y%m%d}_{end_dt:%Y%m%d}"

global_attrs = {
    "description":    "AC__TC__2B tropospheric cloud occurrence (codes 7–21: liquid, rain, snow, ice cloud), v2 processing",
    "date_range":     f"{START_DATE} to {END_DATE}",
    "target_classes": str(TARGET_CLASSES),
    "exclude_codes":  str(EXCLUDE_CODES),
    "max_qc_flag":    MAX_QC_FLAG,
    "grid_resolution_deg": GRID_RES_DEG,
    "min_samples":    MIN_SAMPLES_PER_CELL,
}

# --- 3D occurrence (lat x lon x height) ---
occurrence_3d = np.divide(
    cloud_count_3d, total_count_3d,
    out=np.full_like(cloud_count_3d, np.nan, dtype=np.float64),
    where=total_count_3d > 0,
)
if MIN_SAMPLES_PER_CELL and MIN_SAMPLES_PER_CELL > 0:
    occurrence_3d[total_count_3d < MIN_SAMPLES_PER_CELL] = np.nan

xr.Dataset(
    {"occurrence":   (["latitude", "longitude", "height"], occurrence_3d),
     "cloud_count":  (["latitude", "longitude", "height"], cloud_count_3d),
     "total_count":  (["latitude", "longitude", "height"], total_count_3d)},
    coords={"latitude": lat_centers, "longitude": lon_centers, "height": target_h},
    attrs=global_attrs,
).to_netcdf(f"{outdir}/{_base}_3d.nc")

# --- Lat/lon map (height-collapsed) ---
xr.Dataset(
    {"occurrence":   (["latitude", "longitude"], occurrence_2d),
     "cloud_count":  (["latitude", "longitude"], cloud_2d),
     "total_count":  (["latitude", "longitude"], tot_2d)},
    coords={"latitude": lat_centers, "longitude": lon_centers},
    attrs=global_attrs,
).to_netcdf(f"{outdir}/{_base}_latlon.nc")

# --- Lat/height map (longitude-collapsed) ---
xr.Dataset(
    {"occurrence":   (["latitude", "height"], occurrence_lh),
     "cloud_count":  (["latitude", "height"], cloud_lh),
     "total_count":  (["latitude", "height"], tot_lh)},
    coords={"latitude": lat_centers, "height": target_h},
    attrs=global_attrs,
).to_netcdf(f"{outdir}/{_base}_latheight.nc")

print(f"Saved to {outdir}/")
print(f"  {_base}_3d.nc")
print(f"  {_base}_latlon.nc")
print(f"  {_base}_latheight.nc")